<a href="https://colab.research.google.com/github/Tobi2904/Fine-tune-chatbot/blob/main/TestFineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# Cài đặt Unsloth và các thư viện hỗ trợ
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# Khai báo cấu hình
max_seq_length = 2048 # Độ dài tối đa của 1 đoạn hội thoại (tính bằng token)

# Tải Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # Nén 4-bit để vừa với GPU miễn phí
)

# Gắn Adapter (LoRA) vào mô hình
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Kích thước ma trận phụ
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    bias = "none",
     use_gradient_checkpointing = "unsloth",
 )

In [ ]:
print(model)

In [ ]:
import random
from datasets import load_dataset

# 1. TEMPLATE BẮT BUỘC PHẢI CÓ
alpaca_prompt = """Dưới đây là một chỉ thị mô tả một nhiệm vụ, đi kèm với đầu vào cung cấp thêm bối cảnh. Hãy viết một phản hồi hoàn thành yêu cầu một cách chính xác.

### Chỉ thị:
{}

### Đầu vào:
{}

### Phản hồi:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output)
        text = text + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

# 2. CHỈ TẢI DATA CỦA BẠN (Đã xóa dòng yahma thừa)
dataset = load_dataset("json", data_files = "dataset_tuvan_khachhang.json", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. CHIA RANDOM TRAIN/TEST
current_random_seed = random.randint(0, 999999)
split_dataset = dataset.train_test_split(test_size=0.1, seed=current_random_seed, shuffle=True)

train_data = split_dataset["train"]
test_data = split_dataset["test"]

In [ ]:
len(train_data)
len(test_data)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Khởi tạo Cỗ máy huấn luyện
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = test_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,

        # --- CẤU HÌNH TỰ ĐỘNG CHẤM ĐIỂM ---
        eval_strategy = "steps",
        eval_steps = 10,
        logging_first_step = True,

        # 👇 THÊM DÒNG NÀY VÀO ĐỂ FIX LỖI FP16 👇
        optim = "paged_adamw_8bit",

        output_dir = "outputs",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
    ),
)

print("🚀 Bắt đầu quá trình huấn luyện (Training)...")
trainer.train()

In [ ]:
from transformers import TextStreamer

# 1. BẬT CHẾ ĐỘ SUY LUẬN (INFERENCE)
# Lệnh này cực kỳ quan trọng: Báo cho PyTorch biết "Học xong rồi, giờ chỉ trả lời thôi, tắt hết tính toán đạo hàm đi cho nhẹ RAM"
FastLanguageModel.for_inference(model)

# 2. CHUẨN BỊ CÂU HỎI TEST (ĐÓNG VAI ỨNG VIÊN KHÓ TÍNH)
test_instruction = "Bạn là một chuyên viên tư vấn. Hãy xử lý tình huống sau."
test_input = "Tôi mua hàng về dùng 2 ngày đã hỏng rồi. Chất lượng kém quá!"

# Dùng lại đúng template lúc học, nhưng để trống phần Output {} để AI tự điền
prompt = alpaca_prompt.format(test_instruction, test_input, "")

# 3. MÃ HÓA CÂU HỎI
inputs = tokenizer(
    [prompt],
    return_tensors = "pt"
).to("cuda")

# Cài đặt bộ gõ chữ từ từ (như ChatGPT)
text_streamer = TextStreamer(tokenizer, skip_prompt=True) # skip_prompt=True để màn hình chỉ in câu trả lời, không in lại câu hỏi

print("🤖 AI HR đang trả lời...\n")
print("-" * 50)

# 4. YÊU CẦU AI SINH CÂU TRẢ LỜI
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,  # Giới hạn độ dài câu trả lời (tránh nó nói luyên thuyên không dừng)
    temperature = 0.3,     # Độ sáng tạo: 0.3 là mức lý tưởng cho HR (chuyên nghiệp, điềm đạm, không bịa đặt)
    use_cache = True
)

In [ ]:
# 1. Lưu trọng số LoRA và bộ mã hóa (Tokenizer) vào thư mục
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 2. Nén thư mục đó lại thành file zip để tải về máy không bị lỗi
!zip -r lora_model.zip lora_model/
print("🎉 Đã nén xong! Hãy vào thư mục bên góc trái Colab để tải file lora_model.zip về máy.")

In [ ]:
from google.colab import files
files.download('lora_model.zip')